# 📝 Word Embeddings — Student Workbook
## Practice Exercises & Assignments

---

**Instructions**
- All cells marked with `# YOUR CODE HERE` require you to fill in an implementation.
- All cells marked with `# YOUR ANSWER HERE (Markdown)` require a written explanation.
- Cells marked `# DO NOT MODIFY — test cell` are auto-graders: run them to check your work.
- 🏆 **Challenge** exercises are optional but highly recommended.

| Section | Topic | Points |
|---------|-------|--------|
| Part 1 | One-Hot Vectors | 10 |
| Part 2 | Co-Occurrence Matrix | 20 |
| Part 3 | PPMI | 20 |
| Part 4 | Word2Vec — Softmax | 25 |
| Part 5 | Negative Sampling | 25 |
| Part 6 | GloVe Loss | 20 |
| Part 7 | Analogy Arithmetic | 15 |
| Part 8 | Bias Detection | 20 |
| Part 9 | Open-Ended Analysis | 25 |
| **Total** | | **180** |

---

In [ ]:
# ── Setup — Run this cell first ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE
import random, math, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# ── Shared corpus ────────────────────────────────────────────────
CORPUS = [
    "the dog barked at the cat",
    "the cat sat on the mat",
    "the dog and cat are pets",
    "dogs and cats are animals",
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king and queen are royals",
    "the prince will become king",
    "the princess will become queen",
    "man and woman are humans",
    "the man went to the market",
    "the woman went to the market",
    "dogs like to eat meat",
    "cats like to eat fish",
    "the king ate bread and meat",
    "bread and fish are food",
    "animals eat food to survive",
    "humans eat bread and meat and fish",
    "the dog ran in the park",
    "the cat ran in the garden",
    "the prince and princess are royals",
    "the king is a man",
    "the queen is a woman",
    "man has a dog as a pet",
    "woman has a cat as a pet",
    "the market sells bread and fish",
    "animals in the park include dogs and cats",
    "the kingdom has a king and a queen",
    "humans and animals both need food",
    "the prince is a young man",
    "the princess is a young woman",
]

STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'and', 'to', 'in', 'at', 'on', 'as', 'has',
             'both', 'will', 'of', 'have', 'be', 'went', 'ran', 'need', 'include'}

def build_vocab(corpus, stopwords=STOPWORDS, min_freq=2):
    tokens = [w for sent in corpus for w in sent.lower().split()]
    freq   = Counter(tokens)
    words  = sorted([w for w, c in freq.items() if c >= min_freq and w not in stopwords])
    w2i    = {w: i for i, w in enumerate(words)}
    i2w    = {i: w for w, i in w2i.items()}
    return words, w2i, i2w

vocab_words, word2idx, idx2word = build_vocab(CORPUS)
V = len(vocab_words)

print(f"Vocabulary ({V} words): {' | '.join(vocab_words)}")
print("\n✅ Setup complete. Start with Part 1 below.")

---
# Part 1 — One-Hot Vectors (10 pts)

## Exercise 1.1 — Build One-Hot Vectors (4 pts)

Implement `one_hot(word, word2idx, V)` that returns a numpy array of shape `(V,)` with a `1` at the word's index and `0` everywhere else. Return a zero vector for unknown words.

In [ ]:
def one_hot(word, word2idx, V):
    """
    Return the one-hot vector for `word`.
    
    Parameters
    ----------
    word     : str  — the word to encode
    word2idx : dict — vocabulary mapping word → index
    V        : int  — vocabulary size
    
    Returns
    -------
    np.ndarray of shape (V,), dtype float
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Quick visual check (run this after implementing)
# one_hot('cat', word2idx, V)  # Should have a single 1.0

In [ ]:
# DO NOT MODIFY — test cell
v = one_hot('dog', word2idx, V)
assert v.shape == (V,),           "❌ Shape should be (V,)"
assert v.sum() == 1.0,            "❌ Only one 1.0 expected"
assert v[word2idx['dog']] == 1.0, "❌ 1.0 should be at 'dog' index"
v_unk = one_hot('xxxxxxxx', word2idx, V)
assert v_unk.sum() == 0.0,        "❌ Unknown word should give zero vector"
print("✅ Exercise 1.1 passed!")

## Exercise 1.2 — Cosine Similarity Between One-Hot Vectors (3 pts)

Implement `cosine_sim(v1, v2)`. Then compute the cosine similarity between:
- `dog` and `cat` (should be semantically similar)
- `dog` and `king` (should be semantically unrelated)

What do you observe? Why?

In [ ]:
def cosine_sim(v1, v2):
    """
    Compute cosine similarity between two vectors.
    cos(v1, v2) = (v1 · v2) / (||v1|| * ||v2||)
    Return 0.0 if either vector is zero.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Test your function
# sim_dog_cat  = cosine_sim(one_hot('dog', word2idx, V), one_hot('cat', word2idx, V))
# sim_dog_king = cosine_sim(one_hot('dog', word2idx, V), one_hot('king', word2idx, V))
# print(f"dog ↔ cat  : {sim_dog_cat}")
# print(f"dog ↔ king : {sim_dog_king}")

In [ ]:
# DO NOT MODIFY — test cell
v1 = np.array([1, 0, 0], dtype=float)
v2 = np.array([0, 1, 0], dtype=float)
v3 = np.array([1, 0, 0], dtype=float)
assert abs(cosine_sim(v1, v2) - 0.0) < 1e-6,  "❌ Orthogonal vectors should give 0"
assert abs(cosine_sim(v1, v3) - 1.0) < 1e-6,  "❌ Identical vectors should give 1"
assert abs(cosine_sim(v1, np.zeros(3))) < 1e-6,"❌ Zero vector should give 0"
print("✅ Exercise 1.2 passed!")

## Exercise 1.3 — Written Reflection (3 pts)

Based on your results above, answer:
1. What is the cosine similarity between ANY two different words using one-hot encoding?
2. Why is this a fundamental problem for NLP?
3. What property would an ideal word representation have instead?

**YOUR ANSWER HERE (Markdown)**

1. ...

2. ...

3. ...

---
# Part 2 — Co-Occurrence Matrix (20 pts)

## Exercise 2.1 — Build the Co-Occurrence Matrix (10 pts)

Implement `build_cooccurrence(corpus, word2idx, window)`. 

**Algorithm:**
- For each sentence, tokenize and keep only vocabulary words
- For each word at position `i`, look at all words at positions `i-window` to `i+window` (excluding position `i` itself)
- Increment `M[center_idx, context_idx]` by 1 for each such pair

In [ ]:
def build_cooccurrence(corpus, word2idx, window=2):
    """
    Build a V × V co-occurrence count matrix.
    M[i, j] = number of times word_i appears within `window` positions of word_j.
    
    Parameters
    ----------
    corpus   : list of str  — list of sentences
    word2idx : dict         — word → integer index
    window   : int          — context window size (look ±window positions)
    
    Returns
    -------
    np.ndarray of shape (V, V), dtype float32
    """
    V = len(word2idx)
    M = np.zeros((V, V), dtype=np.float32)
    
    # YOUR CODE HERE
    # Hint: for each sentence, split and filter; then loop over positions
    raise NotImplementedError

    return M


# Build and inspect
# M_raw = build_cooccurrence(CORPUS, word2idx, window=2)
# print(f"Matrix shape: {M_raw.shape}")
# print(f"dog ↔ cat count: {M_raw[word2idx['dog'], word2idx['cat']]}")
# print(f"dog ↔ king count: {M_raw[word2idx['dog'], word2idx['king']]}")

In [ ]:
# DO NOT MODIFY — test cell
M_raw = build_cooccurrence(CORPUS, word2idx, window=2)
assert M_raw.shape == (V, V),  "❌ Shape should be (V, V)"
assert M_raw.sum() > 0,        "❌ Matrix should have non-zero entries"
# Symmetry: M[i,j] == M[j,i] (co-occurrence is symmetric)
assert np.allclose(M_raw, M_raw.T, atol=0.5), "❌ Matrix should be roughly symmetric"
# dog and cat should co-occur more than dog and king
if 'dog' in word2idx and 'cat' in word2idx and 'king' in word2idx:
    assert M_raw[word2idx['dog'], word2idx['cat']] >= M_raw[word2idx['dog'], word2idx['king']], \
        "❌ dog-cat should co-occur at least as much as dog-king"
print("✅ Exercise 2.1 passed!")

## Exercise 2.2 — Nearest Neighbours from Co-Occurrence (6 pts)

Implement `nearest_neighbours(query_word, M, word2idx, idx2word, topk=5)` that:
1. Retrieves the co-occurrence vector for `query_word`
2. L2-normalises it
3. Returns the top-k most similar words by cosine similarity (exclude the word itself)

In [ ]:
def nearest_neighbours(query_word, M, word2idx, idx2word, topk=5):
    """
    Find the top-k nearest neighbours of `query_word` in embedding matrix M.
    Similarity measure: cosine similarity (dot product of L2-normalised vectors).
    
    Returns: list of (word, similarity_score) sorted by descending similarity.
    Returns [] if word not in vocabulary.
    """
    if query_word not in word2idx:
        print(f"'{query_word}' not in vocabulary")
        return []
    
    # YOUR CODE HERE
    # Step 1: Get the query vector and normalise it
    # Step 2: Normalise all rows of M
    # Step 3: Compute dot products (cosine similarities)
    # Step 4: Exclude the query word itself and return top-k
    raise NotImplementedError


# Test:
# for w in ['dog', 'king', 'bread']:
#     nn = nearest_neighbours(w, M_raw, word2idx, idx2word, topk=5)
#     print(f"  '{w}' → {nn}")

In [ ]:
# DO NOT MODIFY — test cell
nn = nearest_neighbours('king', M_raw, word2idx, idx2word, topk=5)
assert isinstance(nn, list),         "❌ Should return a list"
assert len(nn) == 5,                 "❌ Should return 5 neighbours"
assert isinstance(nn[0], tuple),     "❌ Each element should be a (word, score) tuple"
assert nn[0][1] >= nn[-1][1],        "❌ Should be sorted descending"
assert 'king' not in [w for w,_ in nn], "❌ Should not include the query word itself"
print("✅ Exercise 2.2 passed!")
print("\n'king' nearest neighbours:", nn)

## Exercise 2.3 — Window Size Experiment (4 pts)

Train co-occurrence matrices with `window=1` and `window=4`. Compare nearest neighbours for `'king'`. What changes and why?

*(Hint: smaller window → more syntactic similarity; larger window → more topical similarity)*

In [ ]:
# YOUR CODE HERE
# 1. Build M_win1 = build_cooccurrence(CORPUS, word2idx, window=1)
# 2. Build M_win4 = build_cooccurrence(CORPUS, word2idx, window=4)
# 3. Print nearest neighbours for 'king' using both matrices
# 4. Explain the difference in the markdown cell below

raise NotImplementedError

**YOUR ANSWER HERE (Markdown)**

What changed between window=1 and window=4?

...

---
# Part 3 — PPMI (20 pts)

## Exercise 3.1 — Implement PPMI (12 pts)

$$\text{PPMI}(w, c) = \max\left(0,\ \log_2\frac{P(w,c)}{P(w) \cdot P(c)}\right)$$

Where:
- $P(w, c) = \frac{N(w,c)}{\sum_{w,c} N(w,c)}$  (joint probability)
- $P(w) = \frac{\sum_c N(w,c)}{\sum_{w,c} N(w,c)}$  (marginal probability of word w)
- $P(c) = \frac{\sum_w N(w,c)}{\sum_{w,c} N(w,c)}$  (marginal probability of context c)

**Important:** Set entries to 0 where the denominator would be zero.

In [ ]:
def compute_ppmi(M):
    """
    Compute the Positive PMI matrix from a raw co-occurrence matrix M.
    PPMI(w, c) = max(0, log2(P(w,c) / (P(w) * P(c))))
    
    Parameters
    ----------
    M : np.ndarray of shape (V, V) — raw co-occurrence counts
    
    Returns
    -------
    np.ndarray of shape (V, V), same shape as M, all values >= 0
    """
    # YOUR CODE HERE
    # Hints:
    # - total    = M.sum()             → total co-occurrence events
    # - row_sum  = M.sum(axis=1, keepdims=True)  → P(w) unnormalised
    # - col_sum  = M.sum(axis=0, keepdims=True)  → P(c) unnormalised
    # - Use np.errstate(divide='ignore', invalid='ignore') to suppress log(0) warnings
    # - Use np.maximum(pmi, 0) to get PPMI
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
M_ppmi = compute_ppmi(M_raw)
assert M_ppmi.shape == (V, V),       "❌ Shape should be (V, V)"
assert (M_ppmi >= 0).all(),          "❌ All PPMI values must be >= 0"
# PPMI should be 0 where M_raw is 0
zero_mask = (M_raw == 0)
assert (M_ppmi[zero_mask] == 0).all(), "❌ PPMI must be 0 where M is 0"
# PPMI for frequently co-occurring pairs should be positive
if 'king' in word2idx and 'queen' in word2idx:
    assert M_ppmi[word2idx['king'], word2idx['queen']] > 0, "❌ king-queen PPMI should be positive"
print("✅ Exercise 3.1 passed!")

## Exercise 3.2 — Compare Raw Counts vs PPMI (8 pts)

1. Find the top-3 nearest neighbours for `'king'` using raw counts
2. Find the top-3 nearest neighbours for `'king'` using PPMI
3. Create a side-by-side bar chart showing the similarities
4. Write a 2-sentence explanation of why PPMI gives better results

In [ ]:
# Part A: Nearest neighbours comparison
# YOUR CODE HERE
# nn_raw   = nearest_neighbours('king', M_raw,  word2idx, idx2word, topk=3)
# nn_ppmi  = nearest_neighbours('king', M_ppmi, word2idx, idx2word, topk=3)

raise NotImplementedError

In [ ]:
# Part B: Create the bar chart
# YOUR CODE HERE
# Use matplotlib to create a side-by-side bar chart comparing raw vs PPMI similarities
# Title, x-label, y-label, legend required

raise NotImplementedError

**YOUR ANSWER HERE (Markdown)**

Why does PPMI give better nearest neighbours than raw counts?

...

---
# Part 4 — Word2Vec Softmax (25 pts)

## Exercise 4.1 — Numerically Stable Softmax (5 pts)

Implement `softmax(x)`. **Critical:** subtract the maximum value before exponentiating for numerical stability (prevents overflow).

In [ ]:
def softmax(x):
    """
    Numerically stable softmax.
    softmax(x_i) = exp(x_i - max(x)) / sum(exp(x_j - max(x)))
    
    Parameters
    ----------
    x : np.ndarray of shape (n,)
    
    Returns
    -------
    np.ndarray of shape (n,), all values in (0,1), sums to 1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
x1 = np.array([1.0, 2.0, 3.0])
p  = softmax(x1)
assert p.shape == (3,),            "❌ Shape should match input"
assert abs(p.sum() - 1.0) < 1e-6, "❌ Output should sum to 1"
assert (p > 0).all(),              "❌ All probabilities should be positive"
assert p[2] > p[1] > p[0],        "❌ Higher input → higher probability"
# Numerical stability test with very large values
x_large = np.array([1000.0, 1001.0, 999.0])
p_large = softmax(x_large)
assert not np.any(np.isnan(p_large)), "❌ Should handle large values without NaN"
print("✅ Exercise 4.1 passed!")

## Exercise 4.2 — Word2Vec Loss & Gradients (12 pts)

Implement the forward pass and gradient computation for one (center, context) pair.

**Loss:** $J = -\log P(o|c) = -u_o^T v_c + \log \sum_w \exp(u_w^T v_c)$

**Gradients:**
- $\frac{\partial J}{\partial v_c} = \hat{y} - y$ (predicted distribution minus one-hot true label) — expressed as: $\sum_w p_w u_w - u_o$
- $\frac{\partial J}{\partial u_w} = (p_w - \mathbf{1}[w=o]) \cdot v_c$

Where $\hat{y} = softmax(U v_c)$ is the predicted probability distribution and $y$ is the one-hot target.

In [ ]:
def w2v_loss_and_grads(center_idx, context_idx, W_center, W_context):
    """
    Compute the Word2Vec Skip-Gram loss and gradients for ONE (center, context) pair.
    
    Parameters
    ----------
    center_idx  : int — index of the center word
    context_idx : int — index of the true context word
    W_center    : np.ndarray of shape (V, d) — center word embedding matrix
    W_context   : np.ndarray of shape (V, d) — context word embedding matrix
    
    Returns
    -------
    loss         : float — the negative log-likelihood loss
    grad_vc      : np.ndarray of shape (d,)  — gradient w.r.t. center word vector
    grad_U       : np.ndarray of shape (V, d) — gradient w.r.t. ALL context vectors
    """
    V, d = W_center.shape
    
    # YOUR CODE HERE
    # Step 1: Get v_c and compute scores = W_context @ v_c  (shape: V)
    # Step 2: Compute probs = softmax(scores)  (shape: V)
    # Step 3: loss = -log(probs[context_idx])
    # Step 4: grad_vc = (probs @ W_context) - W_context[context_idx]
    # Step 5: delta = probs.copy(); delta[context_idx] -= 1
    #         grad_U = outer product of delta and v_c
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
d_test = 5
Wc  = np.random.randn(V, d_test) * 0.01
Wu  = np.random.randn(V, d_test) * 0.01
c_i = word2idx.get('king', 0)
o_i = word2idx.get('queen', 1)

loss, gc, gU = w2v_loss_and_grads(c_i, o_i, Wc, Wu)

assert isinstance(loss, (float, np.floating)), "❌ Loss should be a float"
assert loss > 0,                    "❌ Loss should be positive"
assert gc.shape == (d_test,),       "❌ grad_vc shape should be (d,)"
assert gU.shape == (V, d_test),     "❌ grad_U shape should be (V, d)"
assert not np.isnan(loss),          "❌ Loss should not be NaN"

# Gradient check: numerical gradient should match analytical gradient
eps = 1e-4
num_grad_vc = np.zeros(d_test)
for j in range(d_test):
    Wc_plus  = Wc.copy(); Wc_plus[c_i, j]  += eps
    Wc_minus = Wc.copy(); Wc_minus[c_i, j] -= eps
    loss_p, _, _ = w2v_loss_and_grads(c_i, o_i, Wc_plus,  Wu)
    loss_m, _, _ = w2v_loss_and_grads(c_i, o_i, Wc_minus, Wu)
    num_grad_vc[j] = (loss_p - loss_m) / (2 * eps)
assert np.allclose(gc, num_grad_vc, atol=1e-4), f"❌ Gradient check failed! Analytical: {gc[:3]}, Numerical: {num_grad_vc[:3]}"
print("✅ Exercise 4.2 passed (including gradient check)!")

## Exercise 4.3 — Training Loop (8 pts)

Complete the training loop for Word2Vec Skip-Gram. The structure is given; fill in the update steps.

In [ ]:
def generate_training_pairs(corpus, word2idx, window=2):
    """Generate (center_idx, context_idx) training pairs."""
    pairs = []
    for sentence in corpus:
        words   = sentence.lower().split()
        idx_seq = [word2idx[w] for w in words if w in word2idx]
        for i, ci in enumerate(idx_seq):
            for j in range(max(0, i-window), min(len(idx_seq), i+window+1)):
                if i != j:
                    pairs.append((ci, idx_seq[j]))
    return pairs


def train_skipgram(training_pairs, V, d=20, lr=0.05, epochs=200):
    """
    Train Word2Vec Skip-Gram using full softmax.
    
    Returns
    -------
    W_center  : np.ndarray (V, d) — trained center word vectors
    W_context : np.ndarray (V, d) — trained context word vectors
    loss_hist : list of float     — average loss per epoch
    """
    W_center  = np.random.randn(V, d) * 0.01
    W_context = np.random.randn(V, d) * 0.01
    loss_hist = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        shuffled   = training_pairs.copy()
        random.shuffle(shuffled)
        
        for center_idx, context_idx in shuffled:
            # YOUR CODE HERE
            # 1. Call w2v_loss_and_grads to get loss, grad_vc, grad_U
            # 2. Add loss to total_loss
            # 3. Update W_center[center_idx] using gradient descent
            # 4. Update W_context using gradient descent
            pass
        
        avg_loss = total_loss / max(len(shuffled), 1)
        loss_hist.append(avg_loss)
        if epoch % 50 == 0 or epoch == epochs-1:
            print(f"  Epoch {epoch+1:4d}/{epochs}  Loss: {avg_loss:.4f}")
    
    return W_center, W_context, loss_hist


# Train
training_pairs = generate_training_pairs(CORPUS, word2idx, window=2)
print(f"Training pairs: {len(training_pairs)}")
print("Training Skip-Gram...")
# Uncomment to run:
# W_c, W_ctx, loss_hist = train_skipgram(training_pairs, V, d=20, lr=0.05, epochs=200)
# E_sg = normalize(W_c, norm='l2')

In [ ]:
# DO NOT MODIFY — test cell
# Run the training (this may take ~30 seconds)
try:
    W_c, W_ctx, loss_hist = train_skipgram(training_pairs, V, d=20, lr=0.05, epochs=200)
    E_sg = normalize(W_c, norm='l2')
    assert len(loss_hist) == 200,     "❌ Should have 200 loss values"
    assert loss_hist[-1] < loss_hist[0] * 0.95, "❌ Loss should decrease during training"
    assert E_sg.shape == (V, 20),     "❌ Embedding matrix shape should be (V, 20)"
    print("✅ Exercise 4.3 passed!")
except NotImplementedError:
    print("❌ Please implement the training loop first")

---
# Part 5 — Negative Sampling (25 pts)

## Exercise 5.1 — Sigmoid Function (3 pts)

Implement the sigmoid function with numerical stability (clip inputs to prevent overflow).

In [ ]:
def sigmoid(x):
    """
    Numerically stable sigmoid: σ(x) = 1 / (1 + exp(-x))
    Clip x to [-30, 30] to prevent overflow.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
assert abs(sigmoid(0)   - 0.5)  < 1e-6, "❌ sigmoid(0) should be 0.5"
assert sigmoid(1000)   > 0.99,          "❌ sigmoid(large) should be close to 1"
assert sigmoid(-1000)  < 0.01,          "❌ sigmoid(-large) should be close to 0"
assert not np.isnan(sigmoid(np.array([1000.0]))).any(), "❌ No NaN for large inputs"
print("✅ Exercise 5.1 passed!")

## Exercise 5.2 — Negative Sampling Loss & Gradients (12 pts)

$$J_{neg} = -\log\sigma(u_{ctx}^T v_c) - \sum_{k=1}^{K} \log\sigma(-u_{neg_k}^T v_c)$$

**Gradient w.r.t. $v_c$:**
- Positive term: $-(1 - \sigma(u_{ctx}^T v_c)) \cdot u_{ctx}$
- Negative terms: $\sum_k (1 - \sigma(-u_{neg_k}^T v_c)) \cdot u_{neg_k}$

**Gradient w.r.t. $u_{ctx}$:** $-(1 - \sigma(u_{ctx}^T v_c)) \cdot v_c$

**Gradient w.r.t. each $u_{neg_k}$:** $(1 - \sigma(-u_{neg_k}^T v_c)) \cdot v_c$

In [ ]:
def neg_samp_loss_and_grads(center_idx, context_idx, neg_indices, W_center, W_context):
    """
    Compute negative sampling loss and gradients for one training step.
    
    Parameters
    ----------
    center_idx  : int  — center word index
    context_idx : int  — positive (true) context word index
    neg_indices : list of int — K negative sample indices
    W_center    : np.ndarray (V, d)
    W_context   : np.ndarray (V, d)
    
    Returns
    -------
    loss        : float
    grad_vc     : np.ndarray (d,)          — grad w.r.t. center vector
    grad_u_pos  : np.ndarray (d,)          — grad w.r.t. positive context vector
    grad_u_neg  : dict {int: np.ndarray}   — grads w.r.t. each negative context vector
    """
    v_c   = W_center[center_idx]
    u_pos = W_context[context_idx]
    
    # YOUR CODE HERE
    # Positive pair:
    #   score_pos = dot(u_pos, v_c)
    #   sig_pos   = sigmoid(score_pos)
    #   loss      = -log(sig_pos + 1e-9)
    #   grad_vc   = -(1 - sig_pos) * u_pos    (initialise here)
    #   grad_u_pos= -(1 - sig_pos) * v_c
    #
    # Negative pairs (loop over neg_indices):
    #   score_neg = dot(W_context[ni], v_c)
    #   sig_neg   = sigmoid(-score_neg)    [note the minus sign!]
    #   loss     += -log(sig_neg + 1e-9)
    #   grad_vc  += (1 - sig_neg) * W_context[ni]
    #   grad_u_neg[ni] = (1 - sig_neg) * v_c
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
d_t  = 4
Wc_t = np.random.randn(V, d_t) * 0.1
Wu_t = np.random.randn(V, d_t) * 0.1
c_i  = word2idx.get('king', 0)
o_i  = word2idx.get('queen', 1)
neg  = [word2idx.get('bread', 2), word2idx.get('fish', 3)]

loss_ns, gc_ns, gup, gun = neg_samp_loss_and_grads(c_i, o_i, neg, Wc_t, Wu_t)
assert isinstance(loss_ns, (float, np.floating)), "❌ Loss should be float"
assert loss_ns > 0,               "❌ Loss should be positive"
assert gc_ns.shape == (d_t,),     "❌ grad_vc should be shape (d,)"
assert gup.shape  == (d_t,),      "❌ grad_u_pos should be shape (d,)"
assert isinstance(gun, dict),     "❌ grad_u_neg should be a dict"
assert len(gun) == 2,             "❌ Should have one entry per negative sample"
print("✅ Exercise 5.2 passed!")

## Exercise 5.3 — The 3/4 Power Sampling Distribution (10 pts)

Implement the negative sampling distribution $P(w) \propto f(w)^{3/4}$, train a model using it, and compare performance against uniform sampling.

In [ ]:
def build_neg_sampler(corpus, word2idx, power=0.75):
    """
    Build a negative sampling probability distribution.
    P(w) ∝ frequency(w)^power
    
    Returns
    -------
    vocab : list of str  — vocabulary words in word2idx order
    probs : np.ndarray   — sampling probabilities for each word
    """
    # YOUR CODE HERE
    # 1. Count word frequencies in corpus (only vocabulary words)
    # 2. Compute counts^power for each word
    # 3. Normalise to get probabilities
    raise NotImplementedError


# Visualise and compare uniform vs 3/4 power
# YOUR CODE HERE
# Create a bar chart comparing:
#   - raw frequency (normalised)
#   - 3/4 power distribution
#   - uniform distribution (1/V for all words)
# What does the 3/4 power do to the distribution?

raise NotImplementedError

**YOUR ANSWER HERE (Markdown)**

What does the 3/4 power exponent do to the sampling distribution, and why is this better than using raw frequencies or uniform sampling?

...

---
# Part 6 — GloVe Loss (20 pts)

## Exercise 6.1 — Implement the GloVe Weighting Function (5 pts)

$$f(x) = \begin{cases} (x / x_{max})^\alpha & \text{if } x < x_{max} \\ 1 & \text{otherwise} \end{cases}$$

In [ ]:
def glove_weight(x, x_max=100, alpha=0.75):
    """
    GloVe weighting function.
    Works element-wise on numpy arrays.
    f(0) = 0, f(x_max) = 1, f(x > x_max) = 1.
    """
    # YOUR CODE HERE
    # Use np.where(condition, val_if_true, val_if_false)
    raise NotImplementedError


# Plot the weighting function
# YOUR CODE HERE
# Plot f(x) for x in [0, 200] with x_max=100
# The curve should start at 0, grow sublinearly, and plateau at 1

In [ ]:
# DO NOT MODIFY — test cell
assert glove_weight(0)   == 0.0,    "❌ f(0) should be 0"
assert glove_weight(100) == 1.0,    "❌ f(x_max) should be 1"
assert glove_weight(200) == 1.0,    "❌ f(>x_max) should be 1"
assert 0 < glove_weight(50) < 1,    "❌ f(0 < x < x_max) should be between 0 and 1"
# Test with array input
x_arr = np.array([0, 50, 100, 200])
w_arr = glove_weight(x_arr)
assert w_arr.shape == (4,),          "❌ Should work element-wise on arrays"
print("✅ Exercise 6.1 passed!")

## Exercise 6.2 — Implement GloVe Training Step (15 pts)

Implement one training step for GloVe. The objective for one word pair (i, j):

$$J_{ij} = f(N_{ij}) \cdot \left(v_i^T \tilde{v}_j + b_i + \tilde{b}_j - \log N_{ij}\right)^2$$

Gradients:
- $\frac{\partial J_{ij}}{\partial v_i} = 2 f(N_{ij}) \cdot r_{ij} \cdot \tilde{v}_j$
- $\frac{\partial J_{ij}}{\partial \tilde{v}_j} = 2 f(N_{ij}) \cdot r_{ij} \cdot v_i$
- $\frac{\partial J_{ij}}{\partial b_i} = \frac{\partial J_{ij}}{\partial \tilde{b}_j} = 2 f(N_{ij}) \cdot r_{ij}$

Where $r_{ij} = v_i^T \tilde{v}_j + b_i + \tilde{b}_j - \log N_{ij}$

In [ ]:
def glove_step(i, j, N_ij, W, U, bw, bu, lr=0.05):
    """
    Perform one GloVe gradient update for word pair (i, j).
    Updates W, U, bw, bu IN PLACE.
    
    Parameters
    ----------
    i, j  : int   — word indices
    N_ij  : float — co-occurrence count N(i, j)
    W     : (V, d) — word vectors (updated in place)
    U     : (V, d) — context vectors (updated in place)
    bw    : (V,)   — word biases (updated in place)
    bu    : (V,)   — context biases (updated in place)
    lr    : float  — learning rate
    
    Returns: loss for this pair
    """
    if N_ij == 0:
        return 0.0
    
    # YOUR CODE HERE
    # 1. fij = glove_weight(N_ij)
    # 2. r   = dot(W[i], U[j]) + bw[i] + bu[j] - log(N_ij)
    # 3. loss = fij * r^2
    # 4. Compute gradients and update W[i], U[j], bw[i], bu[j]
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY — test cell
d_t  = 3
W_t  = np.random.randn(V, d_t) * 0.1
U_t  = np.random.randn(V, d_t) * 0.1
bw_t = np.zeros(V)
bu_t = np.zeros(V)

W_before = W_t.copy()
loss_g   = glove_step(0, 1, 5.0, W_t, U_t, bw_t, bu_t, lr=0.05)

assert isinstance(loss_g, (float, np.floating)), "❌ Should return a float loss"
assert loss_g >= 0,                              "❌ Loss should be non-negative"
assert not np.allclose(W_t[0], W_before[0]),     "❌ W[i] should be updated"
assert glove_step(0, 1, 0.0, W_t, U_t, bw_t, bu_t) == 0.0, "❌ N=0 should give 0 loss"
print("✅ Exercise 6.2 passed!")

---
# Part 7 — Analogy Arithmetic (15 pts)

## Exercise 7.1 — Implement Analogy Solver (8 pts)

Implement `solve_analogy(a, b, c, embeddings, word2idx, idx2word, topk=3)` that solves:
> "a is to b as c is to ___?"

Method: find $d^* = \arg\max_{d \neq a,b,c} \cos(v_d,\ v_b - v_a + v_c)$

In [ ]:
def solve_analogy(a, b, c, embeddings, word2idx, idx2word, topk=3):
    """
    Solve analogy: a is to b as c is to ?
    Returns top-k candidates (excluding a, b, c themselves).
    
    Parameters
    ----------
    a, b, c    : str
    embeddings : np.ndarray (V, d) — L2-normalised
    
    Returns
    -------
    list of (word, score) sorted by descending score
    """
    # Check all words exist
    for w in [a, b, c]:
        if w not in word2idx:
            print(f"'{w}' not in vocabulary")
            return []
    
    # YOUR CODE HERE
    # 1. Get va, vb, vc
    # 2. Compute query = vb - va + vc
    # 3. Normalise query
    # 4. Compute cosine similarities = embeddings @ query
    # 5. Exclude a, b, c and return top-k
    raise NotImplementedError


# Test (after training E_sg in Part 4)
# results = solve_analogy('man', 'king', 'woman', E_sg, word2idx, idx2word)
# print("man is to king as woman is to:", results)

## Exercise 7.2 — Analogy Geometry Visualisation (7 pts)

Create a 2D visualisation (using PCA to project to 2D) that shows the parallelogram formed by `man`, `woman`, `king`, `queen`. Draw arrows to show the semantic direction vectors.

In [ ]:
# YOUR CODE HERE
# 1. Get the 4 word vectors from E_sg (or any trained embeddings)
# 2. Project to 2D using PCA
# 3. Plot each word as a labelled point
# 4. Draw arrows:
#    - man → king   (royalty direction for males)
#    - woman → queen (royalty direction for females)
#    - man → woman  (gender direction for humans)
#    - king → queen (gender direction for royals)
# 5. These should form a (approximate) parallelogram!

raise NotImplementedError

---
# Part 8 — Bias Detection (20 pts)

## Exercise 8.1 — Gender Direction (8 pts)

Implement `find_bias_direction(E, word2idx, pair_a, pair_b)` that estimates the gender bias direction as the normalised difference between two gender-defining words.

In [ ]:
def find_bias_direction(embeddings, word2idx, word_a='man', word_b='woman'):
    """
    Estimate the gender direction as the unit vector from word_a to word_b.
    direction = (v_b - v_a) / ||v_b - v_a||
    
    Returns None if either word is missing from vocabulary.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def measure_bias(embeddings, word2idx, vocab_words, direction):
    """
    Measure how much each word is projected onto the bias direction.
    Positive = closer to word_b (woman), Negative = closer to word_a (man).
    
    Returns dict: {word: projection_score}
    """
    # YOUR CODE HERE
    # For each word in vocab_words that's in word2idx,
    # compute dot(embeddings[word2idx[word]], direction)
    raise NotImplementedError

## Exercise 8.2 — Debiasing (12 pts)

Implement `debias_embeddings(E, word2idx, neutral_words, direction)` that removes the gender component from neutral words by subtracting their projection onto the gender direction.

$$v_w^{\text{debiased}} = v_w - (v_w \cdot g) \cdot g$$

Where $g$ is the gender direction (a unit vector).

In [ ]:
def debias_embeddings(embeddings, word2idx, neutral_words, direction):
    """
    Remove the bias direction component from embeddings of neutral words.
    Gendered words (man, woman, king, queen, etc.) are NOT modified.
    
    Parameters
    ----------
    embeddings   : np.ndarray (V, d)  — original embeddings
    word2idx     : dict
    neutral_words: list of str — words to debias
    direction    : np.ndarray (d,)    — unit bias direction vector
    
    Returns: new np.ndarray (V, d) with neutral words debiased and re-normalised
    """
    # YOUR CODE HERE
    # 1. Copy embeddings
    # 2. For each neutral word: E[i] = E[i] - dot(E[i], direction) * direction
    # 3. Re-normalise all vectors
    raise NotImplementedError


# Evaluate: plot bias projections before and after debiasing
# YOUR CODE HERE
# 1. Get the gender direction
# 2. Measure bias before debiasing
# 3. Debias the embeddings
# 4. Measure bias after debiasing
# 5. Create a before/after bar chart

raise NotImplementedError

---
# Part 9 — Open-Ended Analysis (25 pts)

Choose **TWO** of the following challenges. Each is worth 12.5 points (2 × 12.5 = 25 total).

---

### Challenge A — FastText-style Subword Embeddings (12.5 pts)

Implement a simplified FastText model:
1. For each word, extract all character n-grams (e.g., `"cat"` → `"<ca"`, `"cat"`, `"at>"`, `"<cat>"` for n=3)
2. The word's embedding = mean of its n-gram embeddings
3. Train using the same Skip-Gram objective
4. Test: can the model represent out-of-vocabulary words (e.g., `"doggy"`)?

In [ ]:
# CHALLENGE A — FastText Subword Embeddings
# YOUR CODE HERE

def get_char_ngrams(word, n=3):
    """
    Get all character n-grams for a word, including boundary markers.
    Example: get_char_ngrams('cat', n=3) → ['<ca', 'cat', 'at>', '<cat>']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def build_ngram_vocab(corpus, word2idx, n=3):
    """
    Build a vocabulary of all character n-grams appearing in the corpus.
    Returns: ngram2idx dict
    """
    # YOUR CODE HERE
    raise NotImplementedError


# After implementing above, train a FastText-style model and:
# - Show that 'doggy' (OOV word) gets a reasonable embedding
# - Compare nearest neighbours for 'dogs' between standard W2V and FastText

---
### Challenge B — Semantic Change Detection (12.5 pts)

1. Create two versions of the corpus: `corpus_a` (original) and `corpus_b` (with `'king'` used in food contexts)
2. Train separate Word2Vec models on each
3. Implement both change detection methods (alignment + neighbour intersection)
4. Show which words are detected as having shifted meaning

In [ ]:
# CHALLENGE B — Semantic Change Detection
# YOUR CODE HERE

CORPUS_MODIFIED = CORPUS.copy()
# Add sentences that use 'king' in a food/brand context
CORPUS_MODIFIED += [
    # YOUR NEW SENTENCES HERE
]

def semantic_displacement(E1, E2, word2idx):
    """
    Align E2 to E1 using Orthogonal Procrustes, then measure
    1 - cosine_similarity for each word.
    Higher = more semantic change.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def neighbour_change_score(E1, E2, word_idx, k=5):
    """
    Measure change as 1 - |intersection of k-NN in E1 and E2| / k
    """
    # YOUR CODE HERE
    raise NotImplementedError

---
### Challenge C — Cross-Lingual Alignment (12.5 pts)

1. Simulate a "second language" by rotating your Word2Vec embeddings + adding noise
2. Implement Orthogonal Procrustes to learn the alignment matrix
3. Show: given `dog` in language 1, can you retrieve the correct translation `chien` (same word, different coordinates) in language 2?
4. Plot before/after alignment for 6 word pairs

In [ ]:
# CHALLENGE C — Cross-Lingual Alignment
# YOUR CODE HERE

def orthogonal_procrustes(X, Z):
    """
    Find rotation W = argmin ||ZW - X||_F  s.t. W^T W = I
    Closed form: W = U V^T where Z^T X = U Σ V^T
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cross_lingual_retrieval(query_idx, E_src_aligned, E_tgt, topk=3):
    """
    Find the nearest neighbours in E_tgt for the query in E_src_aligned.
    Returns list of (idx, similarity) pairs.
    """
    # YOUR CODE HERE
    raise NotImplementedError

---
# 🏁 Final Reflection (Mandatory — 5 bonus pts)

Answer all three questions thoughtfully.

**1. Which method (PPMI+SVD, Word2Vec, GloVe) performed best on your toy corpus, and why do you think that is?**

YOUR ANSWER:
...

---

**2. The distributional hypothesis says "you shall know a word by the company it keeps." Give one concrete example from your experiments where this worked well, and one where it failed or gave a surprising result.**

YOUR ANSWER:
...

---

**3. Static word embeddings (Word2Vec, GloVe) assign a single fixed vector to each word regardless of context. What is the limitation of this? (Hint: think about the word "bank.")**

YOUR ANSWER:
...

---
# 📋 Self-Assessment Checklist

Run this cell to check which exercises are complete.

In [ ]:
# ── Self-Assessment ──────────────────────────────────────────────
print("Word Embeddings Workbook — Self-Assessment")
print("="*55)

tests = [
    ("Ex 1.1 one_hot",         lambda: one_hot('dog', word2idx, V).sum() == 1.0),
    ("Ex 1.2 cosine_sim",      lambda: abs(cosine_sim(np.array([1.,0.,0.]), np.array([0.,1.,0.]))) < 1e-6),
    ("Ex 2.1 build_cooccurrence", lambda: build_cooccurrence(CORPUS[:5], word2idx, window=1).shape == (V,V)),
    ("Ex 2.2 nearest_neighbours", lambda: isinstance(nearest_neighbours('king', M_raw, word2idx, idx2word), list)),
    ("Ex 3.1 compute_ppmi",    lambda: (compute_ppmi(M_raw) >= 0).all()),
    ("Ex 4.1 softmax",         lambda: abs(softmax(np.array([1.,2.,3.])).sum() - 1.0) < 1e-6),
    ("Ex 4.2 w2v_loss_and_grads", lambda: isinstance(w2v_loss_and_grads(0, 1, np.random.randn(V,4)*0.01, np.random.randn(V,4)*0.01)[0], (float, np.floating))),
    ("Ex 5.1 sigmoid",         lambda: abs(sigmoid(0) - 0.5) < 1e-6),
    ("Ex 5.2 neg_samp_loss",   lambda: isinstance(neg_samp_loss_and_grads(0,1,[2,3],np.random.randn(V,4)*0.01,np.random.randn(V,4)*0.01)[0], (float, np.floating))),
    ("Ex 5.3 build_neg_sampler", lambda: abs(build_neg_sampler(CORPUS, word2idx)[1].sum()-1.0) < 1e-6),
    ("Ex 6.1 glove_weight",    lambda: glove_weight(0)==0 and glove_weight(100)==1.0),
    ("Ex 6.2 glove_step",      lambda: glove_step(0,1,5.0,np.random.randn(V,3)*0.1,np.random.randn(V,3)*0.1,np.zeros(V),np.zeros(V)) >= 0),
]

passed = 0
for name, fn in tests:
    try:
        ok = fn()
        status = "✅ PASS" if ok else "❌ FAIL"
        if ok: passed += 1
    except NotImplementedError:
        status = "⬜ NOT IMPLEMENTED"
    except Exception as e:
        status = f"❌ ERROR: {type(e).__name__}"
    print(f"  {status:<30}  {name}")

print()
print(f"Core exercises passed: {passed}/{len(tests)}")